%% [markdown]
# ESM-2 embedding inference parity — CPU / CUDA / Trainium  (INTEGRATION mode)
Runs ESM-2 as a **feature extractor** (per-residue embeddings; MLM head dropped) on every available
device and checks the extracted embeddings match. CPU is the reference; `cuda` auto-skips when absent;
`neuron` runs on the Trainium core. Pin a free core:
`NEURON_RT_VISIBLE_CORES=0 jupyter nbconvert --to notebook --execute 01_inference_parity.ipynb`.

In [1]:
# %%
import os
os.environ.setdefault("NEURON_RT_VISIBLE_CORES", "0")
import sys
sys.path.insert(0, os.path.abspath("../src"))
import torch
import torch.nn.functional as F
import esm2_reference as R

[W708 23:24:11.206874707 OperatorEntry.cpp:208] Warning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::gather.out(Tensor self, int dim, Tensor index, *, bool sparse_grad=False, Tensor(a!) out) -> Tensor(a!)
    registered at /pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: PrivateUse1
  previous kernel: registered at /pytorch/build/aten/src/ATen/RegisterCPU_3.cpp:7637
       new kernel: registered at NeuronDispatcher.cpp:0 (function operator())


In [2]:
def devices():
    devs = ["cpu"]
    if torch.cuda.is_available():
        devs.append("cuda")
    try:
        import torch_neuronx  # noqa: F401
        devs.append("neuron")
    except Exception as e:
        print("neuron unavailable:", e)
    return devs

In [3]:
DEVICES = devices()
print("torch", torch.__version__, "| devices:", DEVICES, "| model:", R.MODEL_NAME)

torch 2.11.0+cpu | devices: ['cpu', 'neuron'] | model: facebook/esm2_t6_8M_UR50D


In [4]:
# %% [markdown]
# ## Run the integration forward (embeddings + downstream logits) on each device
# %%
def run(device):
    model = R.load(device)
    ids, mask = R.build_inputs()
    with torch.no_grad():
        out = model(ids.to(device), mask.to(device))
    if device == "neuron":
        import torch_neuronx; torch_neuronx.synchronize()
    return tuple(t.detach().float().cpu() for t in out)

In [5]:
results = {d: run(d) for d in DEVICES}
for name, t in zip(R.OUTPUT_ORDER, results["cpu"]):
    print(f"cpu {name:12s} shape={tuple(t.shape)}")

/home/user/miniconda3/envs/torch-neuron/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 13965.95it/s]


[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 13827.38it/s]


[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cpu embeddings   shape=(2, 64, 320)
cpu logits       shape=(2, 2)


In [6]:
# %% [markdown]
# ## Check every device matches CPU (embeddings are the extracted feature)
# %%
def cos(a, b): return F.cosine_similarity(a.flatten(), b.flatten(), dim=0).item()

In [7]:
ref, all_ok = results["cpu"], True
for d in DEVICES:
    if d == "cpu":
        continue
    print(f"\n{d} vs cpu:")
    for name, a, b in zip(R.OUTPUT_ORDER, ref, results[d]):
        c = cos(a, b); mab = (a - b).abs().max().item(); ok = c >= 0.99
        all_ok = all_ok and ok
        print(f"  {name:12s} cosine={c:.6f}  max-abs={mab:.3e}  {'OK' if ok else 'FAIL'}")


neuron vs cpu:
  embeddings   cosine=1.000000  max-abs=4.292e-06  OK
  logits       cosine=1.000000  max-abs=1.192e-07  OK


In [8]:
print("\nEMBEDDING INFERENCE PARITY:", "PASS" if all_ok else "FAIL")
assert all_ok, "embeddings diverged across devices"


EMBEDDING INFERENCE PARITY: PASS
